In [2]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


In [5]:
X = pd.read_csv("../data/secom.data", sep=" ", header=None)
y = pd.read_csv("../data/secom_labels.data", sep =" ", header=None,
                names =["label", "timestamp"])
y["timestamp"] = pd.to_datetime(y["timestamp"], format="%d/%m/%Y %H:%M:%S")

# the label 0/1 ( 1 = fail) and sort by time
df = X.copy()
df["label"]= (y["label"] == 1).astype(int)
df["timestamp"] = y["timestamp"]
df = df.sort_values("timestamp").reset_index(drop=True)

# time  based split
# 80% for training 
# 20% for testing
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]

feature_cols = [c for c in df.columns if c not in ("label", "timestamp")]
X_train, y_train = train[feature_cols], train["label"]
X_test, y_test = test[feature_cols], test["label"]

# drop useless columns
keep = [c for c in feature_cols
        if X_train[c].isna().mean() <= 0.5 and X_train[c].nunique() > 1]
X_train, X_test = X_train[keep], X_test[keep]
print(f"features: {len(feature_cols)} -> {len(keep)}")


# fill missing values with the training median
# scale
imputer =  SimpleImputer(strategy="median")
scaler = StandardScaler()
X_train_p = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_p = scaler.transform(imputer.transform(X_test))

print("fail rate  train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))



features: 590 -> 444
fail rate  train: 0.069  test: 0.054
